# AgriAgent: Gemma 4 Native Function Calling & Multimodal Vision Demo

This notebook demonstrates how **Gemma 4**'s native function calling capabilities are utilized in the **AgriAgent** system to integrate IoT soil sensors, localized weather forecasting, and market crop rates. This upgraded demo showcases **Parallel Function Calling** (running multiple tools in one turn) and **Multimodal Visual Diagnostics** (detecting leaf disease patterns). This serves as the **Clonable Notebook** submission option for the TFUG Prayagraj Hackathon.

### System Architecture
1. **User Query**: Farmer asks a question in plain natural language (e.g., "Water my wheat field if moisture is low and check market price of Wheat").
2. **Gemma 4 Thought**: Model analyzes the prompt and decides to fetch sensor telemetry and market price data in parallel.
3. **Tool Dispatching**: Model issues a structured JSON tool call payload containing **multiple function calls** concurrently.
4. **API Execution**: The local system parses the tool calls, executes them, and returns all results in parallel.
5. **Final Output**: Gemma 4 synthesizes the data and generates a clear, actionable agricultural response.

## 1. Defining Agricultural Tool APIs
Below are the mock APIs simulating IoT farm sensors, crop visual analysis, and market pricing.

In [ ]:
import json
from datetime import datetime

# Mock databases representing fields in Prayagraj
FIELDS_DB = {
    "field-a": {
        "name": "North Field",
        "crop": "Rice",
        "moisture": 72,
        "ph": 6.2,
        "npk": {"N": 45, "P": 30, "K": 55}
    },
    "field-b": {
        "name": "West Terrace",
        "crop": "Wheat",
        "moisture": 34, # Low moisture -> Needs water
        "ph": 6.8,
        "npk": {"N": 20, "P": 15, "K": 40}
    }
}

MARKET_PRICES = {
    "Wheat": {"price": 2450, "trend": "up"},
    "Rice": {"price": 2200, "trend": "up"}
}

DISEASE_DB = {
    "wheat-rust": {
        "status": "infected",
        "diagnosis": "Stem/Black Rust (Puccinia graminis)",
        "organic": "Apply neem oil extract spray.",
        "chemical": "Apply Propiconazole fungicide immediately."
    }
}

def get_soil_sensors(field_id: str) -> str:
    """Fetches real-time moisture %, temperature, and NPK levels for a field."""
    field_id = field_id.lower().strip()
    if field_id in FIELDS_DB:
        return json.dumps({"status": "success", "field_id": field_id, "data": FIELDS_DB[field_id]})
    return json.dumps({"status": "error", "message": f"Field {field_id} not found."})

def get_market_prices(crop_name: str) -> str:
    """Fetches current market price trends in INR per quintal."""
    crop = crop_name.strip().capitalize()
    if crop in MARKET_PRICES:
        return json.dumps({"status": "success", "crop": crop, "data": MARKET_PRICES[crop]})
    return json.dumps({"status": "error", "message": f"Crop {crop_name} not found."})

def analyze_crop_image(image_id: str) -> str:
    """Simulates Gemma 4 vision-based crop disease diagnosis on leaf photos."""
    img_id = image_id.lower().strip()
    if img_id in DISEASE_DB:
        return json.dumps({"status": "success", "image_id": image_id, "data": DISEASE_DB[img_id]})
    return json.dumps({"status": "success", "image_id": image_id, "data": {"status": "healthy", "diagnosis": "No abnormalities detected."}})

## 2. Gemma 4 Tool Declarations
We declare the tools schema including multiple parameters to demonstrate **Parallel Calling** capabilities.

In [ ]:
SYSTEM_PROMPT = """
You are AgriAgent, an autonomous agricultural AI assistant powered by Gemma 4.
You support parallel function calling. If the user asks a compound question, generate all relevant tool call payloads in a single turn.

Tools Available:
1. get_soil_sensors(field_id: string)
2. get_market_prices(crop_name: string)
3. analyze_crop_image(image_id: string)

Example Output format for parallel tool calls:
Thinking: The user wants to check sensor telemetry and check grain market price. I will call both get_soil_sensors and get_market_prices in parallel.
```json
{
    "tool_calls": [
        {
            "name": "get_soil_sensors",
            "arguments": {"field_id": "field-b"}
        },
        {
            "name": "get_market_prices",
            "arguments": {"crop_name": "Wheat"}
        }
    ]
}
```
"""

## 3. Simulating Parallel Execution Loop
Here we execute compound queries to demonstrate parallel calling.

In [ ]:
def simulate_parallel_agent_loop(user_query: str):
    print(f"[USER QUERY]: {user_query}\n")
    
    # --- Gemma Step 1: Receives Query, Decides Tool Call ---
    print("--- STEP 1: Gemma 4 Thought & Parallel JSON Generation ---")
    
    # Simulating parallel model thought generation
    if "sensors" in user_query and "price" in user_query:
        thought = "The farmer wants to inspect Field B sensors and check Wheat market rates. I will trigger parallel tool executions to get both data points concurrently."
        tool_call = {
            "tool_calls": [
                {
                    "name": "get_soil_sensors",
                    "arguments": {"field_id": "field-b"}
                },
                  {
                    "name": "get_market_prices",
                    "arguments": {"crop_name": "Wheat"}
                }
            ]
        }
    else:
        print("Simulate direct dialogue...")
        return
        
    print(f"Thinking: {thought}")
    print(f"Generated Tool Call JSON:\n{json.dumps(tool_call, indent=4)}\n")
    
    # --- System Step 2: Executes Concurrently ---
    print("--- STEP 2: Local System Parallel Tools Execution ---")
    api_responses = {}
    for call in tool_call["tool_calls"]:
        name = call["name"]
        args = call["arguments"]
        if name == "get_soil_sensors":
            api_responses[name] = get_soil_sensors(args["field_id"])
        elif name == "get_market_prices":
            api_responses[name] = get_market_prices(args["crop_name"])
            
    print(f"Parallel Executions Completed. Results:\n{json.dumps(api_responses, indent=4)}\n")
    
    # --- Gemma Step 3: Final Synthesis ---
    print("--- STEP 3: Gemma 4 Final Synthesis Response ---")
    sensors = json.loads(api_responses["get_soil_sensors"])["data"]
    prices = json.loads(api_responses["get_market_prices"])["data"]
    
    final_answer = (
        f"I queried your crop sensors and the grain markets in parallel.\n\n"
        f"* **Soil Moisture (Field B)**: {sensors['moisture']}% (Low - optimal is >50%)\n"
        f"* **Nutrients (Field B)**: NPK levels are {sensors['npk']['N']} N, {sensors['npk']['P']} P, {sensors['npk']['K']} K\n"
        f"* **Wheat Market Price**: ₹{prices['price']} per Quintal (Trend is {prices['trend']})\n\n"
        f"*Recommendation*: Crop moisture is low. I recommend watering Field B. Market prices are currently rising, so it is highly profitable to prepare your crop harvest."
    )
    print(f"[AGRIAGENT FINAL RECOMMENDATION]:\n{final_answer}")

## 4. Run Parallel Execution Demonstration

In [ ]:
simulate_parallel_agent_loop("Check Field B sensors and tell me the price of Wheat")

## 5. Simulating Multimodal Leaf Vision Analysis

In [ ]:
def simulate_vision_agent_loop(image_id: str):
    print(f"[SIMULATING IMAGE SCAN FOR image_id='{image_id}']\n")
    
    print("--- STEP 1: Gemma 4 Multimodal Input Parsing ---")
    print("Thinking: User provided leaf image crop. I will trigger analyze_crop_image diagnostic model to extract pixel patterns.")
    tool_call = {
        "tool_calls": [
            {
                "name": "analyze_crop_image",
                "arguments": {"image_id": image_id}
            }
        ]
    }
    print(f"Generated Tool Call JSON:\n{json.dumps(tool_call, indent=4)}\n")
    
    print("--- STEP 2: Running Vision Diagnostic API ---")
    api_response = analyze_crop_image(image_id)
    print(f"API Returned: {api_response}\n")
    
    print("--- STEP 3: Gemma 4 Treatment Synthesis ---")
    diag = json.loads(api_response)["data"]
    final_answer = (
        f"**Multimodal Image Diagnostic Complete**\n"
        f"* **Leaf Diagnosis**: {diag['diagnosis']}\n"
        f"* **Organic Treatment**: {diag['organic']}\n"
        f"* **Chemical Treatment**: {diag['chemical']}"
    )
    print(f"[AGRIAGENT FINAL RECOMMENDATION]:\n{final_answer}")

In [ ]:
simulate_vision_agent_loop("wheat-rust")